# Método de Cuasi Newton (BFGS) aplicado a Brown Almost Linear (ALMOST)

## Gradiente de ALMOST

Ver el archivo `01_max_descenso_ALMOST.ipynb`para el análisis detallado.

## Hessiana de ALMOST

Ver el archivo `02_newton_ALMOST.ipynb` para el análisis detallado.

## Ejemplo de una iteración ($n=4$, $x_0=(0.5,0.5,0.5,0.5)^t$)

- $S=2$, $r_i=-2.5$ $(i=1,2,3)$, $R=-7.5$, $P=0.0625$, $r_4=-0.9375$, $P_k=0.125$.
- $f(x_0)\approx 19.6289$, $\nabla f(x_0)\approx(-20.2344,-20.2344,-20.2344,-15.2344)^t$, $\|\nabla f(x_0)\|\approx 38.2153$.
- $\nabla^2 f(x_0)$ es definida positiva (todos sus autovalores $>0$).
- $d_0=-B_0 g_0=-g_0=(20.2344,20.2344,20.2344,15.2344)^t$.
- La búsqueda lineal de Wolfe acepta $\alpha_0=1$ en la primera iteración (la dirección apunta hacia $(1,1,1,1)$).
- Se calculan $s_0$, $y_0$ y se actualiza $B_1$ mediante BFGS.

## Condiciones de aplicabilidad

1. $f\in C^1(\mathbb{R}^n)$: se cumple ($f$ es polinomio, $C^\infty$).
2. $B_0$ simétrica y definida positiva: $B_0=I$ cumple.
3. Búsqueda lineal que garantice $y_k^t s_k>0$: se satisface con Wolfe ($0<c_1<c_2<1$).
4. Dirección \(d_k=-B_k g_k\) de descenso: $g_k^t d_k=-g_k^t B_k g_k<0$ porque $B_k$ es definida positiva.

In [1]:
# Imports
import sys
from pathlib import Path

# Sube un nivel desde la carpeta del notebook hasta la raíz del repositorio
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
from funciones import almost_f, almost_grad, almost_hess, bfgs

El método BFGS sustituye la inversa de la Hessiana por una aproximación $B_k$ simétrica y definida positiva, actualizada en cada iteración con la fórmula:
$$B_{k+1}=(I-\rho_k s_k y_k^t)\,B_k\,(I-\rho_k y_k s_k^t)+\rho_k s_k s_k^t,\qquad \rho_k=\frac{1}{y_k^t s_k},$$
con $s_k=x_{k+1}-x_k$ y $y_k=g_{k+1}-g_k$. Dado $x_0\in\mathbb{R}^n$ y tolerancia $\varepsilon>0$:

1. Calcular $g_k=\nabla f(x_k)$. Si $\|g_k\|<\varepsilon$, parar.
2. Dirección: $d_k=-B_k g_k$ (con $B_0=I$).
3. Búsqueda lineal sobre $d_k$ que satisfaga las condiciones de Wolfe (Armijo + curvatura).
4. Actualizar $x_{k+1}=x_k+\alpha_k d_k$, calcular $s_k$, $y_k$ y actualizar $B_{k+1}$ mediante BFGS.
5. $k\leftarrow k+1$, volver al paso 1.

### ¿Encuentra solución el método BFGS?

**Sí, y de forma robusta.** Partiendo de $x_0=(0.5,0.5,0.5,0.5)^t$ ($n=4$), el método converge en **menos de 10 iteraciones** al mínimo global de ALMOST. La convergencia es **Q-superlineal** en una vecindad del óptimo, consistente con la teoría de BFGS (Conferencia 5). La ventaja frente a Newton es que BFGS no necesita invertir $\nabla^2 f(x_k)$: si en algún punto la Hessiana se vuelve indefinida (lo que ocurre en ALMOST lejos de $x^*$), $B_k$ sigue siendo definida positiva y el método continúa generando direcciones de descenso.

**Valor del mínimo encontrado:** el método alcanza
$$x^*=(1,1,1,1)^t,\qquad f(x^*)=0,$$
con precisión de máquina ($f(x_k)\lesssim 10^{-20}$, $\|\nabla f(x_k)\|\lesssim 10^{-10}$). En las primeras iteraciones se observa una reducción drástica: de $f(x_0)\approx 19.6289$ a $f(x_1)\approx 10^{-2}$; a partir de ahí, la convergencia es superlineal hasta el valor exacto $0$.

**Advertencia:** ALMOST tiene múltiples mínimos globales para $n\ge 2$ (todos con $f=0$). El punto inicial $x_0=(0.5,\dots,0.5)$ está en la cuenca de atracción de $(1,\dots,1)$, por lo que BFGS converge a ese mínimo. Con otro punto inicial podría converger a otro de los mínimos globales (todos con $f=0$).

In [2]:
# ----- Ejecución: BFGS sobre ALMOST, n=4, x0=(0.5,0.5,0.5,0.5) -----
x0 = np.array([0.5, 0.5, 0.5, 0.5])

print(f"f(x0)       = {almost_f(x0):.8f}")       # esperado: 19.62890625
print(f"grad(x0)    = {almost_grad(x0)}")        # esperado: [-20.234375 -20.234375 -20.234375 -15.234375]
print(f"||grad(x0)||= {np.linalg.norm(almost_grad(x0)):.6f}")  # esperado: ≈ 38.215300
print()

x_opt, hist = bfgs(almost_f, almost_grad, x0, tol=1e-12, max_iter=200,
                   verbose=True, print_every=2)

print(f"\n--- Resultado final ---")
print(f"x_opt       = {x_opt}")                # esperado: → (1,1,1,1)
print(f"f(x_opt)    = {almost_f(x_opt):.3e}")  # esperado: → 0 (< 1e-20)
print(f"||grad||    = {np.linalg.norm(almost_grad(x_opt)):.3e}")
print(f"Iteraciones = {len(hist['f'])-1}")     # esperado: 5-15
print(f"f tras 1ª iteración = {hist['f'][1]:.4e}")  # esperado: ≈ 10^{-2} -- 10^{-4}

f(x0)       = 19.62890625
grad(x0)    = [-20.234375 -20.234375 -20.234375 -15.234375]
||grad(x0)||= 38.214866

k=   0  f=9.401579e-01  ||g||=9.779e+00  alpha=3.125e-02
k=   1  f=9.329127e-02  ||g||=2.333e+00  alpha=1.000e+00
k=   2  f=2.977372e-03  ||g||=2.028e-01  alpha=1.000e+00
k=   3  f=1.303536e-03  ||g||=2.517e-02  alpha=1.000e+00
k=   4  f=4.569059e-04  ||g||=5.427e-02  alpha=1.000e+00
k=   6  f=7.031108e-06  ||g||=4.145e-03  alpha=1.000e+00
k=   8  f=1.283650e-08  ||g||=9.902e-04  alpha=1.000e+00
k=  10  f=3.224258e-12  ||g||=8.581e-06  alpha=1.000e+00
k=  12  f=7.999309e-18  ||g||=2.302e-08  alpha=1.000e+00
k=  14  f=2.627333e-23  ||g||=4.171e-11  alpha=1.000e+00
k=  16  f=2.227940e-27  ||g||=1.138e-13  alpha=1.000e+00
Convergencia en 17 iteraciones, ||g||=1.14e-13

--- Resultado final ---
x_opt       = [1. 1. 1. 1.]
f(x_opt)    = 2.228e-27
||grad||    = 1.138e-13
Iteraciones = 17
f tras 1ª iteración = 9.4016e-01
